**Meeting minutes inquiry RAG**

This project is an AI-powered assistant that answers questions about council meeting minutes. 

It uses a vector database to retrieve relevant context from the minutes and a language model to generate concise, accurate responses based strictly on the retrieved content.


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from datasets import load_dataset
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr



load_dotenv(override=True)
api_keys = os.getenv("OPENAI_API_KEY")

openai = OpenAI(api_key=api_keys, base_url="https://openrouter.ai/api/v1")
DB="meeting_vector_db"


In [ ]:
!pip install langchain_huggingface
!pip install unstructured

In [ ]:
# folder to store meeting minutes
output_dir = "meeting_minutes"
os.makedirs(output_dir, exist_ok=True)

# load dataset
dataset = load_dataset("huuuyeah/meetingbank", split="train")

# take first 40
subset = dataset.select(range(40))

# save each meeting into its own file
for i, item in enumerate(subset):
    text = item["transcript"] 

    file_path = f"{output_dir}/meeting_{i+1}.txt"

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)



In [ ]:
# load documents
loader = DirectoryLoader("meeting_minutes", glob="*.txt")
documents = loader.load()

# split documents
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = splitter.split_documents(documents)

# embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# create vector store
vectorstore = Chroma.from_documents(
    docs,
    embeddings,
    persist_directory="meeting_vector_db"
)

vectorstore.persist()

print("Vector database created.")

In [ ]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, base_url="https://openrouter.ai/api/v1", api_key=api_keys)

In [ ]:
context=retriever.invoke("What did councilman espinosa say")

In [ ]:
system_prompt_template = """
You are a helpful assistant that answers questions using retrieved meeting minutes from a vector database.

Use only the provided meeting context to answer the question.
If the answer is present in the meeting minutes, summarize it clearly and concisely.
If a name is mentioned, make sure to include it in the response.
If the information is not in the provided context, say so

Do not make up facts.
Base your response strictly on the retrieved meeting minutes.
When possible, reference key decisions, action items, or discussions mentioned in the meeting notes.
Context:
{context}
"""

In [ ]:
def inquiry(question: str, history):
    chunks = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in chunks)
    system_prompt = system_prompt_template.format(context=context)
    messages = [SystemMessage(content=system_prompt)] + history + [HumanMessage(content=question)]
    response = llm.invoke(messages)
    return response.content


In [ ]:
gr.ChatInterface(inquiry).launch()